In [2]:
import pickle
import pandas as pd 
import numpy as np
import tensorflow
from tensorflow.keras.models import load_model


In [3]:
with open('labelencoder_gender.pkl','rb')as file :
    lb_gender=pickle.load(file)
with open('OneHotEncoder_geo.pkl','rb')as file :
    encoder_geo=pickle.load(file)
with open('scalar.pkl','rb')as file :
    scalar=pickle.load(file)
model=load_model('model.h5')


In [5]:
print(model)
print(encoder_geo)
print(lb_gender)
print(scalar)

<Sequential name=sequential, built=True>
OneHotEncoder()
LabelEncoder()
StandardScaler()


In [6]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [8]:
df=pd.DataFrame([input_data])

In [9]:
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [10]:
df['Gender']=lb_gender.transform(df['Gender'])


In [11]:
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [13]:
geo_array=encoder_geo.transform(df[['Geography']])

In [15]:
geo_array.toarray()

array([[1., 0., 0.]])

In [16]:
geo_df=pd.DataFrame(geo_array.toarray(),columns=encoder_geo.get_feature_names_out())

In [17]:
geo_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [18]:
geo_df=geo_df.reset_index(drop=True)
df=df.reset_index(drop=True)

In [19]:
new_df=pd.concat([df,geo_df],axis=1)

In [20]:
new_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [22]:
new_df.drop('Geography',axis=1,inplace=True)

In [23]:
new_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [24]:
new_df=scalar.transform(new_df)

In [25]:
new_df

array([[-0.52493139,  0.91255717,  0.09471269, -0.69884427, -0.26134615,
         0.79931788,  0.64007158,  0.97530483, -0.87196872,  0.99451504,
        -0.57658047, -0.57388614]])

In [26]:
new_df.shape

(1, 12)

In [27]:
pred=model.predict(new_df)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 555ms/step


In [28]:
pred

array([[0.03178964]], dtype=float32)

In [29]:
pred_prob=pred[0][0]

In [30]:
if pred_prob > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
